# TD1 — Temporal data contract and forecasting baselines — Core

**Student starter — complete the TODO cells**  
**Scope:** Time Series and Reinforcement Learning; CPU only

## Working and claim contract

Local Python 3.11/3.12 with the locked `uv` environment is the primary path. Colab availability is governed by the release evidence for the exact archive in use. Public base seeds are `11, 23, 37, 51, 71`. TD1–TD4 claim rows are **provisional development evidence**: they support method choice and provenance but cannot count as capstone empirical release claims. Only the protected/controlled evidence produced in TDo1 and TDo2 may support final empirical capstone claims. Restart and run top-to-bottom before handoff.

## Learning contract

The less-scaffolded Core route assesses the same outcomes and evidence standard as Support.

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd

def locate_course_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate/'uv.lock').is_file() and (candidate/'shared').is_dir(): return candidate.resolve()
    for lock in start.glob('*/uv.lock'):
        if (lock.parent/'shared').is_dir(): return lock.parent.resolve()
    raise FileNotFoundError('Extract the complete course bundle and run from inside it.')
COURSE_ROOT=locate_course_root(); os.chdir(COURSE_ROOT)
if str(COURSE_ROOT) not in sys.path: sys.path.insert(0,str(COURSE_ROOT))

: 

In [ ]:
import shared
print('Course root:',COURSE_ROOT)
print('Primary path: locked local CPU; check the release evidence before using Colab')


## 1 — Temporal contract and anti-leakage invariants

Load only public development data; record its hash and decision contract. Implement row/date/target/censoring invariants plus a failing negative test.

In [ ]:
import hashlib, numpy as np, pandas as pd
import sklearn
from sklearn.metrics import mean_absolute_error
from shared.data import dataset_path
from shared.evaluation import smape
path=dataset_path('daily_demand_development.csv')
demand=pd.read_csv(path,parse_dates=['date'])
data_sha256=hashlib.sha256(path.read_bytes()).hexdigest()
print(demand.shape,data_sha256[:16])
forecast_contract={"date_column": "date",
    "target_column": "demand_units",
    "frequency": "daily",
    "date_unique": True,
    "date_sorted": True,
    "target_non_negative": True,
    "censoring_column": "stockout_flag",
    "censoring_value": 1} # TODO
def assert_daily_contract(frame):
    # Vérifier la présence des colonnes obligatoires
    required_columns = [
        "date",
        "demand_units",
        "promotion",
        "holiday_period",
        "stockout_flag"
    ]

    assert all(col in frame.columns for col in required_columns), \
        "Missing required columns"

    # Vérifier que le DataFrame n'est pas vide
    assert len(frame) > 0, "Dataset is empty"

    # Vérifier les dates
    assert frame["date"].notna().all(), \
        "Missing dates"

    assert frame["date"].is_unique, \
        "Duplicate dates detected"

    assert frame["date"].is_monotonic_increasing, \
        "Dates are not sorted"

    # Vérifier que les dates sont consécutives
    date_differences = frame["date"].diff().dropna()

    assert (date_differences == pd.Timedelta(days=1)).all(), \
        "Dates are not consecutive daily observations"

    # Vérifier la variable cible
    assert frame["demand_units"].notna().all(), \
        "Missing target values"

    assert np.isfinite(frame["demand_units"]).all(), \
        "Target contains non-finite values"

    assert (frame["demand_units"] >= 0).all(), \
        "Negative demand detected"

    # Vérifier les variables binaires
    for col in ["promotion", "holiday_period", "stockout_flag"]:

        assert frame[col].notna().all(), \
            f"Missing values in {col}"

        assert frame[col].isin([0, 1]).all(), \
            f"Invalid binary values in {col}"
    # TODO
    pass
assert_daily_contract(demand)

print("Canonical dataset: VALID")

# Test 2 : créer volontairement une erreur
broken_demand = demand.copy()

# Introduire une date dupliquée
broken_demand.loc[1, "date"] = broken_demand.loc[0, "date"]

# Vérifier que notre fonction détecte l'erreur
try:
    assert_daily_contract(broken_demand)

except AssertionError as error:
    print("Negative test: PASSED")
    print("Detected error:", error)

else:
    raise AssertionError("Negative test failed: invalid data was accepted")
# TODO: canonical and deliberately broken calls

## 2 — Baselines and evidence

Freeze the final 28 public development days. Implement historical mean, last value, and lag-7 seasonal naive from earlier history only; compare MAE/sMAPE on aligned dates.

In [ ]:
train=demand.iloc[:-28].copy(); validation=demand.iloc[-28:].copy(); assert train.date.max()<validation.date.min()
def baseline_forecasts(train_values,horizon=28,season=7):
    train_values = np.asarray(train_values, dtype=float)

    # Vérifier que les données sont suffisantes
    assert len(train_values) >= season, "Not enough historical data"

    assert np.isfinite(train_values).all(), \
        "Training data contains invalid values"

    # 1. Historical Mean
    historical_mean = np.full(
        horizon,
        np.mean(train_values)
    )

    # 2. Last Value
    last_value = np.full(
        horizon,
        train_values[-1]
    )

    # 3. Seasonal Naive (lag 7)
    seasonal_naive = np.resize(
        train_values[-season:],
        horizon
    )
    return historical_mean, last_value, seasonal_naive
# Générer les prévisions à partir de l'historique
forecasts = baseline_forecasts(
    train["demand_units"].values,
    horizon=len(validation),
    season=7
)

# Associer chaque prévision à son modèle
forecast_names = [
    "Historical Mean",
    "Last Value",
    "Seasonal Naive"
]

forecast_dict = dict(zip(forecast_names, forecasts))

# Comparer les performances sur les mêmes dates
y_true = validation["demand_units"].values

results = []

for name, prediction in forecast_dict.items():

    mae = mean_absolute_error(y_true, prediction)
    smape_score = smape(y_true, prediction)

    results.append({
        "Model": name,
        "MAE": mae,
        "sMAPE": smape_score
    })

# Tableau comparatif
scores = pd.DataFrame(results)

display(scores)

# Visualisation des prévisions
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(
    validation["date"],
    y_true,
    label="Actual Demand",
    marker="o"
)

for name, prediction in forecast_dict.items():

    plt.plot(
        validation["date"],
        prediction,
        label=name,
        linestyle="--"
    )

plt.title("Comparison of Forecasting Baselines")
plt.xlabel("Date")
plt.ylabel("Demand Units")
plt.legend()
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show() # TODO: common comparison and figure
scores

## Required interpretation

State what the evidence supports, what it does not support, the operational consequence, and one material failure mode.

In [ ]:
# TODO: replace with 4–6 evidence-backed sentences.
# Identify the best model according to MAE
best_model = scores.loc[scores["MAE"].idxmin(), "Model"]
best_mae = scores["MAE"].min()

# Identify the best model according to sMAPE
best_smape_model = scores.loc[scores["sMAPE"].idxmin(), "Model"]
best_smape = scores["sMAPE"].min()

interpretation = (
    f"The comparison of the three forecasting baselines shows that "
    f"{best_model} achieves the lowest MAE ({best_mae:.2f}), "
    f"while {best_smape_model} achieves the lowest sMAPE ({best_smape:.2f}). "
    f"These results support the use of simple forecasting methods as "
    f"development benchmarks on the 28-day validation period. "
    f"However, they do not guarantee future forecasting performance or "
    f"prove that the selected model will generalize to other periods. "
    f"Operationally, the best-performing baseline can serve as a reference "
    f"for evaluating more advanced forecasting models. "
    f"A major limitation is stockout-related censoring, because observed "
    f"demand may underestimate actual demand when products are unavailable."
)
print(interpretation or 'Interpretation pending.')

: 

## Optional Extension — ungraded

Freeze valid Core first. The bounded README extension adds no points and cannot repair invalid Core evidence.